In [1]:
# Cell 1: Imports and Device Setup
import os
import time
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:

# Cell 2: Color Map and Mask Conversion (Background = index 0)
COLOR_MAP = {
    (0, 0, 0): 0,            # Background to ignore
    (255, 255, 0): 1,        # Crater
    (255, 0, 0): 2,          # Rough
    (0, 255, 0): 3,          # Smooth
    (0, 0, 255): 4           # Alluvial_Fan
}

def mask_to_class(mask: Image.Image) -> np.ndarray:
    """
    Convert an RGB mask image to a 2D array of class indices.
    """
    mask = np.array(mask)
    class_mask = np.zeros((mask.shape[0], mask.shape[1]), dtype=np.int64)
    for rgb, idx in COLOR_MAP.items():
        class_mask[(mask == rgb).all(axis=-1)] = idx
    return class_mask

In [3]:
# Cell 3: Dataset Definition
class MarsDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.image_names = sorted([f for f in os.listdir(image_dir) if f.startswith("img_")])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        mask_name = img_name.replace("img_", "mask_")

        image = Image.open(os.path.join(self.image_dir, img_name)).convert("RGB").resize((256, 256))
        mask_img = Image.open(os.path.join(self.mask_dir, mask_name)).convert("RGB").resize((256, 256))

        if self.transform:
            image = self.transform(image)

        mask = mask_to_class(mask_img)
        mask = torch.tensor(mask, dtype=torch.long)
        return image, mask


In [4]:
# Cell 4: Data Loaders
transform = transforms.Compose([
    transforms.ToTensor(),  # Scales to [0,1]
])

train_dataset = MarsDataset(
    image_dir=r"/kaggle/input/datasets/arjunu312003/mars-datatset/dataset/train_images",
    mask_dir=r"/kaggle/input/datasets/arjunu312003/mars-datatset/dataset/train_masks",
    transform=transform
)
val_dataset = MarsDataset(
    image_dir=r"/kaggle/input/datasets/arjunu312003/mars-datatset/dataset/val_image",
    mask_dir=r"/kaggle/input/datasets/arjunu312003/mars-datatset/dataset/val_mask",
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=8,  shuffle=False, num_workers=0)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class Up(nn.Module):
    """Upscaling then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels // 2, out_channels)  # Fixed this line

    def forward(self, x):
        x = self.up(x)
        x = self.conv(x)  # Added this missing line to apply the conv after upsampling
        return x

class YNet(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True):
        super(YNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        # Encoder (first slanting line)
        self.enc1 = DoubleConv(n_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.enc4 = DoubleConv(256, 512)
        self.pool4 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder 1 (second slanting line, with skip connections)
        self.up1_d1 = Up(1024, 512, bilinear)
        self.dec1_d1 = DoubleConv(1024, 512)  # 512 (skip) + 512
        self.up2_d1 = Up(512, 256, bilinear)
        self.dec2_d1 = DoubleConv(512, 256)
        self.up3_d1 = Up(256, 128, bilinear)
        self.dec3_d1 = DoubleConv(256, 128)
        self.up4_d1 = Up(128, 64, bilinear)
        self.dec4_d1 = DoubleConv(128, 64)
        self.out_d1 = nn.Conv2d(64, n_classes, kernel_size=1)

        # Decoder 2 (straight line, without skip connections)
        self.up1_d2 = Up(1024, 512, bilinear)
        self.dec1_d2 = DoubleConv(512, 512)
        self.up2_d2 = Up(512, 256, bilinear)
        self.dec2_d2 = DoubleConv(256, 256)
        self.up3_d2 = Up(256, 128, bilinear)
        self.dec3_d2 = DoubleConv(128, 128)
        self.up4_d2 = Up(128, 64, bilinear)
        self.dec4_d2 = DoubleConv(64, 64)
        self.out_d2 = nn.Conv2d(64, n_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        e3 = self.enc3(p2)
        p3 = self.pool3(e3)
        e4 = self.enc4(p3)
        p4 = self.pool4(e4)
        b = self.bottleneck(p4)

        # Decoder 1 (with skips)
        u1_d1 = self.up1_d1(b)
        cat1_d1 = torch.cat((e4, u1_d1), dim=1)
        d1_d1 = self.dec1_d1(cat1_d1)
        u2_d1 = self.up2_d1(d1_d1)
        cat2_d1 = torch.cat((e3, u2_d1), dim=1)
        d2_d1 = self.dec2_d1(cat2_d1)
        u3_d1 = self.up3_d1(d2_d1)
        cat3_d1 = torch.cat((e2, u3_d1), dim=1)
        d3_d1 = self.dec3_d1(cat3_d1)
        u4_d1 = self.up4_d1(d3_d1)
        cat4_d1 = torch.cat((e1, u4_d1), dim=1)
        d4_d1 = self.dec4_d1(cat4_d1)
        out1 = self.out_d1(d4_d1)

        # Decoder 2 (straight, no skips)
        u1_d2 = self.up1_d2(b)
        d1_d2 = self.dec1_d2(u1_d2)
        u2_d2 = self.up2_d2(d1_d2)
        d2_d2 = self.dec2_d2(u2_d2)
        u3_d2 = self.up3_d2(d2_d2)
        d3_d2 = self.dec3_d2(u3_d2)
        u4_d2 = self.up4_d2(d3_d2)
        d4_d2 = self.dec4_d2(u4_d2)
        out2 = self.out_d2(d4_d2)

        # Combine outputs (e.g., average for final segmentation)
        final_out = (out1 + out2) / 2

        return final_out

# Additionally, fix the deprecation warning in your training loop:
# Change this:
# with torch.cuda.amp.autocast():
# To this:
# with torch.amp.autocast('cuda'):

In [6]:
num_epochs=100

In [7]:
# Define class weights (higher weight for Alluvial_Fan)
class_weights = torch.tensor([0.0, 1.0, 1.0, 1.0, 1.0])  # [background, Crater, Rough, Smooth, Alluvial_Fan]
class_weights = class_weights.to(device)  # Move to GPU if available

# Custom Combined Loss (CrossEntropy + Dice)
class CombinedLoss(nn.Module):
    def __init__(self, weights, smooth=1e-5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weights, ignore_index=0)
        self.smooth = smooth

    def forward(self, pred, target):
        # CrossEntropyLoss
        ce_loss = self.ce(pred, target)
        # DiceLoss
        pred = pred.softmax(dim=1)
        target_one_hot = torch.nn.functional.one_hot(target, pred.shape[1]).permute(0, 3, 1, 2).float()
        intersection = (pred * target_one_hot).sum(dim=(2, 3))
        union = (pred + target_one_hot).sum(dim=(2, 3))
        dice_loss = 1 - (2 * intersection / (union + self.smooth)).mean()
        return ce_loss + dice_loss  # Combined loss

# Model, Loss, Optimizer, Scheduler
model = YNet(n_channels=3, n_classes=5).to(device)
criterion = CombinedLoss(weights=class_weights)  # Combined loss with class weights [[8]]
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)  # AdamW for regularization [[9]]
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, epochs=num_epochs, steps_per_epoch=len(train_loader)
)
scaler = torch.cuda.amp.GradScaler()  # Mixed precision training [[10]]

/tmp/ipykernel_202/48595935.py:30: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()  # Mixed precision training [[10]]


In [8]:
# Training Loop with Mixed Precision
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for imgs, masks in tqdm(loader, desc="Training", leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():  # Mixed precision [[10]]
            outputs = model(imgs)
            loss = criterion(outputs, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

# Metric helpers (unchanged)
def compute_confusion_elements(pred, true, class_id):
    pred_i = (pred == class_id)
    true_i = (true == class_id)
    tp = (pred_i & true_i).sum()
    fp = (pred_i & ~true_i).sum()
    fn = (~pred_i & true_i).sum()
    return tp, fp, fn

def eval_one_epoch(model, loader, criterion, device, class_ids):
    model.eval()
    total_loss = 0
    sum_tp = {c:0 for c in class_ids}
    sum_fp = {c:0 for c in class_ids}
    sum_fn = {c:0 for c in class_ids}
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Validation", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            total_loss += criterion(logits, masks).item()
            preds = torch.argmax(logits, dim=1)
            for c in class_ids:
                tp, fp, fn = compute_confusion_elements(preds, masks, c)
                sum_tp[c] += tp
                sum_fp[c] += fp
                sum_fn[c] += fn
    loss = total_loss / len(loader)
    ious = {c: sum_tp[c] / (sum_tp[c] + sum_fp[c] + sum_fn[c] + 1e-8) for c in class_ids}
    return loss, ious

In [ ]:
# Cell 8: Training Loop with Per-Class IoU Logging
num_epochs = 100
best_val_iou = 0.0  # Track best IoU for Alluvial_Fan (class 4)
eval_classes = [1, 2, 3, 4]
class_names = {1:'Crater', 2:'Rough', 3:'Smooth', 4:'Alluvial_Fan'}

for epoch in range(1, num_epochs+1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_ious = eval_one_epoch(model, val_loader, criterion, device, eval_classes)
    scheduler.step()

    # Print epoch summary
    print(f"Epoch {epoch}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}")
    for c in eval_classes:
        print(f"    {class_names[c]:15s} IoU: {val_ious[c]:.4f}")

    # Save best model based on Alluvial_Fan IoU
    current_iou = val_ious[4]
    if current_iou > best_val_iou:
        best_val_iou = current_iou
        torch.save(model.state_dict(), "best_mars_segmentation_model.pth")
        print("✅ Saved new best model")

Training:   0%|          | 0/250 [00:00<?, ?it/s]/tmp/ipykernel_202/1868680471.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():  # Mixed precision [[10]]


Epoch 1/100
  Train Loss: 1.7513
  Val   Loss: 1.6759
    Crater          IoU: 0.2328
    Rough           IoU: 0.4666
    Smooth          IoU: 0.5754
    Alluvial_Fan    IoU: 0.0817
✅ Saved new best model


Epoch 2/100
  Train Loss: 1.4678
  Val   Loss: 1.4870
    Crater          IoU: 0.3246
    Rough           IoU: 0.4910
    Smooth          IoU: 0.6425
    Alluvial_Fan    IoU: 0.1034
✅ Saved new best model


Epoch 3/100
  Train Loss: 1.3293
  Val   Loss: 1.3391
    Crater          IoU: 0.4080
    Rough           IoU: 0.4893
    Smooth          IoU: 0.6560
    Alluvial_Fan    IoU: 0.1142
✅ Saved new best model


Epoch 4/100
  Train Loss: 1.2037
  Val   Loss: 1.2273
    Crater          IoU: 0.4536
    Rough           IoU: 0.5718
    Smooth          IoU: 0.6711
    Alluvial_Fan    IoU: 0.1921
✅ Saved new best model


...

Epoch 66/100
  Train Loss: 0.4241
  Val   Loss: 0.5611
    Crater          IoU: 0.8773
    Rough           IoU: 0.8716
    Smooth          IoU: 0.9328
    Alluvial_Fan    IoU: 0.7693


Epoch 67/100
  Train Loss: 0.4263
  Val   Loss: 0.6224
    Crater          IoU: 0.8257
    Rough           IoU: 0.8415
    Smooth          IoU: 0.8699
    Alluvial_Fan    IoU: 0.7213


Epoch 68/100
  Train Loss: 0.4428
  Val   Loss: 0.6013
    Crater          IoU: 0.8523
    Rough           IoU: 0.8399
    Smooth          IoU: 0.8116
    Alluvial_Fan    IoU: 0.6502


Epoch 69/100
  Train Loss: 0.4176
  Val   Loss: 0.5568
    Crater          IoU: 0.8872
    Rough           IoU: 0.8751
    Smooth          IoU: 0.9372
    Alluvial_Fan    IoU: 0.7783
✅ Saved new best model


Training:  96%|█████████▌| 240/250 [05:23<00:13,  1.34s/it]

In [ ]:
# Cell 9: Full Validation Metrics (IoU, Dice, Precision, Recall, Loss, Inference Time)

import time
import numpy as np
import torch

# --- 1. Metric definitions ---
def compute_confusion_elements(pred, true, class_id):
    pred_i = (pred == class_id)
    true_i = (true == class_id)
    tp = np.logical_and(pred_i, true_i).sum()
    fp = np.logical_and(pred_i, ~true_i).sum()
    fn = np.logical_and(~pred_i, true_i).sum()
    return tp, fp, fn

def compute_iou(tp, fp, fn):
    return tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

def compute_dice(tp, fp, fn):
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0

def compute_precision(tp, fp):
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0

def compute_recall(tp, fn):
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

# --- 2. Evaluation loop ---
def evaluate_model(model, dataloader, criterion, num_classes=5, device='cpu'):
    model.eval()
    total_loss = 0.0
    total_time = 0.0
    total_samples = 0

    # accumulators per class
    sum_tp = np.zeros(num_classes, dtype=np.int64)
    sum_fp = np.zeros(num_classes, dtype=np.int64)
    sum_fn = np.zeros(num_classes, dtype=np.int64)

    with torch.no_grad():
        for imgs, masks in dataloader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            # time the forward pass
            torch.cuda.synchronize() if device != 'cpu' else None
            t0 = time.time()
            logits = model(imgs)
            torch.cuda.synchronize() if device != 'cpu' else None
            t1 = time.time()

            batch_size = imgs.size(0)
            total_time += (t1 - t0) * batch_size
            total_samples += batch_size

            # accumulate loss
            loss = criterion(logits, masks)
            total_loss += loss.item() * batch_size

            # predictions & truths to numpy
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            trues = masks.cpu().numpy()

            # accumulate confusion matrix elements
            for c in range(num_classes):
                for p, t in zip(preds, trues):
                    tp, fp, fn = compute_confusion_elements(p, t, c)
                    sum_tp[c] += tp
                    sum_fp[c] += fp
                    sum_fn[c] += fn

    # compute per-class metrics
    ious = [compute_iou(sum_tp[c], sum_fp[c], sum_fn[c]) for c in range(num_classes)]
    dices = [compute_dice(sum_tp[c], sum_fp[c], sum_fn[c]) for c in range(num_classes)]
    precs = [compute_precision(sum_tp[c], sum_fp[c]) for c in range(num_classes)]
    recs  = [compute_recall(sum_tp[c], sum_fn[c]) for c in range(num_classes)]

    return {
        'IoU_per_class': ious,
        'Mean_IoU': np.mean(ious),
        'Dice_per_class': dices,
        'Mean_Dice': np.mean(dices),
        'Precision_per_class': precs,
        'Mean_Precision': np.mean(precs),
        'Recall_per_class': recs,
        'Mean_Recall': np.mean(recs),
        'Avg_Loss': total_loss / total_samples,
        'Avg_Inference_Time_per_Image': total_time / total_samples
    }

# --- 3. Load best model (optional) ---
best_model = YNet(n_channels=3,n_classes=5).to(device)
best_model.load_state_dict(torch.load(r"best_mars_segmentation_model.pth"))
best_model.eval()

# --- 4. Run evaluation ---
metrics = evaluate_model(best_model, val_loader, criterion, num_classes=5, device=device)

# --- 5. Display results ---
print("\n=== Validation Metrics ===")
for cls_id, name in class_names.items():
    print(f"{name:15s} | IoU: {metrics['IoU_per_class'][cls_id]:.4f} "
          f"| Dice: {metrics['Dice_per_class'][cls_id]:.4f} "
          f"| Precision: {metrics['Precision_per_class'][cls_id]:.4f} "
          f"| Recall: {metrics['Recall_per_class'][cls_id]:.4f}")
print(f"\nMean IoU:       {metrics['Mean_IoU']:.4f}")
print(f"Mean Dice:      {metrics['Mean_Dice']:.4f}")
print(f"Mean Precision: {metrics['Mean_Precision']:.4f}")
print(f"Mean Recall:    {metrics['Mean_Recall']:.4f}")
print(f"Avg Loss:       {metrics['Avg_Loss']:.4f}")
print(f"Avg Inference Time per Image: {metrics['Avg_Inference_Time_per_Image']*1000:.2f} ms")


In [ ]:
# Cell 10: Visualize Input | Ground Truth | Predicted Mask

import random
import matplotlib.pyplot as plt
import torch
import numpy as np

# Make sure your CBAMUNet model is loaded and on eval mode
model.eval()

# A helper to map class indices back to RGB for display
INV_COLOR_MAP = {v:k for k,v in COLOR_MAP.items()}

def class_to_rgb(mask):
    """
    mask: 2D numpy array of class indices
    returns: HxWx3 RGB image
    """
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in INV_COLOR_MAP.items():
        rgb[(mask == cls)] = color
    return rgb

def visualize_random_samples(dataset, model, device, n_samples=4):
    """
    Picks n_samples random items from dataset, runs model, and plots.
    """
    indices = random.sample(range(len(dataset)), n_samples)
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 4*n_samples))

    for row, idx in enumerate(indices):
        # load image & gt
        img_tensor, gt_mask = dataset[idx]
        img = img_tensor.permute(1,2,0).numpy()      # HWC, [0,1]
        gt_mask = gt_mask.numpy()                    # HxW ints

        # model prediction
        with torch.no_grad():
            inp = img_tensor.unsqueeze(0).to(device)
            logits = model(inp)
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

        # convert masks to RGB for display
        gt_rgb   = class_to_rgb(gt_mask)
        pred_rgb = class_to_rgb(pred)

        # plot
        axes[row,0].imshow(img)
        axes[row,0].set_title("Input Image")
        axes[row,0].axis('off')

        axes[row,1].imshow(gt_rgb)
        axes[row,1].set_title("Ground Truth")
        axes[row,1].axis('off')

        axes[row,2].imshow(pred_rgb)
        axes[row,2].set_title("Predicted Mask")
        axes[row,2].axis('off')

    plt.tight_layout()
    plt.show()

# Run it:
visualize_random_samples(train_dataset, model, device, n_samples=10)


In [ ]:
# Cell 10: Visualize Input | Ground Truth | Predicted Mask | GT Overlay | Pred Overlay
import matplotlib.pyplot as plt
import torch
import numpy as np

# Make sure your CBAMUNet model is loaded and on eval mode
model.eval()

# A helper to map class indices back to RGB for display
INV_COLOR_MAP = {v: k for k, v in COLOR_MAP.items()}

def class_to_rgb(mask):
    """
    mask: 2D numpy array of class indices
    returns: HxWx3 RGB image (uint8)
    """
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in INV_COLOR_MAP.items():
        rgb[mask == cls] = color
    return rgb

def visualize_first_samples(dataset, model, device, n_samples=10):
    """
    Takes the first n_samples from dataset, runs model, and plots with overlays.
    """
    # Changed: Use first n_samples indices instead of random sampling
    indices = list(range(min(n_samples, len(dataset))))
    
    fig, axes = plt.subplots(n_samples, 5, figsize=(20, 4 * n_samples))
    
    for row, idx in enumerate(indices):
        # Load image & gt
        img_tensor, gt_mask = dataset[idx]
        img = img_tensor.permute(1, 2, 0).numpy()  # HWC, [0,1]
        gt_mask = gt_mask.numpy()                 # HxW ints
        
        # Model prediction
        with torch.no_grad():
            inp = img_tensor.unsqueeze(0).to(device)
            logits = model(inp)
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        
        # Convert masks to RGB for display
        gt_rgb = class_to_rgb(gt_mask)
        pred_rgb = class_to_rgb(pred)
        
        # Prepare overlays (semi-transparent, background class 0 fully transparent)
        h, w = img.shape[:2]
        alpha_value = 0.6
        background_class = 0  # Change this if your background/ignore class is different (e.g., 255)
        
        # Ground truth overlay
        gt_rgb_norm = gt_rgb.astype(np.float32) / 255.0
        alpha_gt = np.full((h, w), alpha_value, dtype=np.float32)
        alpha_gt[gt_mask == background_class] = 0.0
        rgba_gt = np.dstack((gt_rgb_norm, alpha_gt))
        
        # Predicted overlay
        pred_rgb_norm = pred_rgb.astype(np.float32) / 255.0
        alpha_pred = np.full((h, w), alpha_value, dtype=np.float32)
        alpha_pred[pred == background_class] = 0.0
        rgba_pred = np.dstack((pred_rgb_norm, alpha_pred))
        
        # Plot columns
        # 0: Input Image
        axes[row, 0].imshow(img)
        axes[row, 0].set_title(f"Input Image (Sample {idx})")
        axes[row, 0].axis('off')
        
        # 1: Ground Truth Mask
        axes[row, 1].imshow(gt_rgb)
        axes[row, 1].set_title("Ground Truth Mask")
        axes[row, 1].axis('off')
        
        # 2: Predicted Mask
        axes[row, 2].imshow(pred_rgb)
        axes[row, 2].set_title("Predicted Mask")
        axes[row, 2].axis('off')
        
        # 3: Ground Truth Overlay
        axes[row, 3].imshow(img)
        axes[row, 3].imshow(rgba_gt)
        axes[row, 3].set_title("GT Overlay")
        axes[row, 3].axis('off')
        
        # 4: Predicted Overlay
        axes[row, 4].imshow(img)
        axes[row, 4].imshow(rgba_pred)
        axes[row, 4].set_title("Pred Overlay")
        axes[row, 4].axis('off')
    
    plt.tight_layout()
    plt.show()

# Run it with the first 10 samples
visualize_first_samples(train_dataset, model, device, n_samples=10)

In [ ]:
# Cell 10: Visualize Input | Ground Truth | Predicted Mask | GT Overlay | Pred Overlay
import random
import matplotlib.pyplot as plt
import torch
import numpy as np

# Make sure your CBAMUNet model is loaded and on eval mode
model.eval()

# A helper to map class indices back to RGB for display
INV_COLOR_MAP = {v: k for k, v in COLOR_MAP.items()}

def class_to_rgb(mask):
    """
    mask: 2D numpy array of class indices
    returns: HxWx3 RGB image (uint8)
    """
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in INV_COLOR_MAP.items():
        rgb[mask == cls] = color
    return rgb

def visualize_random_samples(dataset, model, device, n_samples=4):
    """
    Picks n_samples random items from dataset, runs model, and plots with overlays.
    """
    indices = random.sample(range(len(dataset)), n_samples)
    fig, axes = plt.subplots(n_samples, 5, figsize=(20, 4 * n_samples))  # Changed to 5 columns
    
    for row, idx in enumerate(indices):
        # Load image & gt
        img_tensor, gt_mask = dataset[idx]
        img = img_tensor.permute(1, 2, 0).numpy()  # HWC, [0,1]
        gt_mask = gt_mask.numpy()                 # HxW ints
        
        # Model prediction
        with torch.no_grad():
            inp = img_tensor.unsqueeze(0).to(device)
            logits = model(inp)
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        
        # Convert masks to RGB for display
        gt_rgb = class_to_rgb(gt_mask)
        pred_rgb = class_to_rgb(pred)
        
        # Prepare overlays (semi-transparent, background class 0 fully transparent)
        h, w = img.shape[:2]
        alpha_value = 0.6
        background_class = 0  # Change this if your background/ignore class is different (e.g., 255)
        
        # Ground truth overlay
        gt_rgb_norm = gt_rgb.astype(np.float32) / 255.0
        alpha_gt = np.full((h, w), alpha_value, dtype=np.float32)
        alpha_gt[gt_mask == background_class] = 0.0
        rgba_gt = np.dstack((gt_rgb_norm, alpha_gt))
        
        # Predicted overlay
        pred_rgb_norm = pred_rgb.astype(np.float32) / 255.0
        alpha_pred = np.full((h, w), alpha_value, dtype=np.float32)
        alpha_pred[pred == background_class] = 0.0
        rgba_pred = np.dstack((pred_rgb_norm, alpha_pred))
        
        # Plot columns
        # 0: Input Image
        axes[row, 0].imshow(img)
        axes[row, 0].set_title("Input Image")
        axes[row, 0].axis('off')
        
        # 1: Ground Truth Mask
        axes[row, 1].imshow(gt_rgb)
        axes[row, 1].set_title("Ground Truth Mask")
        axes[row, 1].axis('off')
        
        # 2: Predicted Mask
        axes[row, 2].imshow(pred_rgb)
        axes[row, 2].set_title("Predicted Mask")
        axes[row, 2].axis('off')
        
        # 3: Ground Truth Overlay
        axes[row, 3].imshow(img)
        axes[row, 3].imshow(rgba_gt)
        axes[row, 3].set_title("GT Overlay")
        axes[row, 3].axis('off')
        
        # 4: Predicted Overlay
        axes[row, 4].imshow(img)
        axes[row, 4].imshow(rgba_pred)
        axes[row, 4].set_title("Pred Overlay")
        axes[row, 4].axis('off')
    
    plt.tight_layout()
    plt.show()

# Run it (with n_samples=10 as in your original code)
visualize_random_samples(train_dataset, model, device, n_samples=10)

In [ ]:
# Cell 11: Visualize Test Images | Predicted Mask | Predicted Overlay
import os
import random
import matplotlib.pyplot as plt
import torch
import numpy as np
from PIL import Image
from torchvision import transforms

# --- 1. Setup ---
test_image_dir = r"/kaggle/input/datasets/arjunu312003/ctx-hirise/Test_CTX/Test_CTX"  # adjust to your test folder
file_names = [f for f in os.listdir(test_image_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))]

# Reuse the inverse color map from before
INV_COLOR_MAP = {v: k for k, v in COLOR_MAP.items()}

def class_to_rgb(mask):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in INV_COLOR_MAP.items():
        rgb[mask == cls] = color
    return rgb

# Image transform (must match training)
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# --- 2. Visualization function ---
def visualize_test_predictions(model, device, n_samples=4):
    model.eval()
    samples = random.sample(file_names, n_samples)
    
    # Changed to 3 columns: Input | Predicted Mask | Predicted Overlay
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 4 * n_samples))
    
    # Overlay settings
    alpha_value = 0.6
    background_class = 0  # Change if your background/ignore class is different (e.g., 255)
    
    for i, fname in enumerate(samples):
        # Load and preprocess image
        path = os.path.join(test_image_dir, fname)
        img_pil = Image.open(path).convert("RGB")
        img_resized = img_pil.resize((256, 256))  # For display (uint8)
        img_tensor = transform(img_pil).unsqueeze(0).to(device)
        
        # Forward pass
        with torch.no_grad():
            logits = model(img_tensor)
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        
        # To display
        img_np = np.array(img_resized)                    # HxWx3 uint8 [0-255]
        img_display = img_np / 255.0                       # Normalize to [0,1] for overlay
        pred_rgb = class_to_rgb(pred)                     # Colored mask uint8
        
        # Prepare predicted overlay (semi-transparent, background fully transparent)
        h, w = pred.shape
        pred_rgb_norm = pred_rgb.astype(np.float32) / 255.0
        alpha_pred = np.full((h, w), alpha_value, dtype=np.float32)
        alpha_pred[pred == background_class] = 0.0
        rgba_pred = np.dstack((pred_rgb_norm, alpha_pred))
        
        # Plot columns
        # 0: Input Image
        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f"Input: {fname}")
        axes[i, 0].axis('off')
        
        # 1: Predicted Mask
        axes[i, 1].imshow(pred_rgb)
        axes[i, 1].set_title("Predicted Mask")
        axes[i, 1].axis('off')
        
        # 2: Predicted Overlay
        axes[i, 2].imshow(img_display)
        axes[i, 2].imshow(rgba_pred)
        axes[i, 2].set_title("Predicted Overlay")
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# --- 3. Run it ---
visualize_test_predictions(model, device, n_samples=10)

In [ ]:
# Cell 11: Visualize Test Images | Predicted Mask | Predicted Overlay
import os
import random
import matplotlib.pyplot as plt
import torch
import numpy as np
from PIL import Image
from torchvision import transforms

# --- 1. Setup ---
test_image_dir = r"/kaggle/input/datasets/arjunu312003/ctx-hirise/Test_HiRISE/Test_HiRISE"  # adjust to your test folder
file_names = [f for f in os.listdir(test_image_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))]

# Reuse the inverse color map from before
INV_COLOR_MAP = {v: k for k, v in COLOR_MAP.items()}

def class_to_rgb(mask):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in INV_COLOR_MAP.items():
        rgb[mask == cls] = color
    return rgb

# Image transform (must match training)
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# --- 2. Visualization function ---
def visualize_test_predictions(model, device, n_samples=4):
    model.eval()
    samples = random.sample(file_names, n_samples)
    
    # Changed to 3 columns: Input | Predicted Mask | Predicted Overlay
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 4 * n_samples))
    
    # Overlay settings
    alpha_value = 0.6
    background_class = 0  # Change if your background/ignore class is different (e.g., 255)
    
    for i, fname in enumerate(samples):
        # Load and preprocess image
        path = os.path.join(test_image_dir, fname)
        img_pil = Image.open(path).convert("RGB")
        img_resized = img_pil.resize((256, 256))  # For display (uint8)
        img_tensor = transform(img_pil).unsqueeze(0).to(device)
        
        # Forward pass
        with torch.no_grad():
            logits = model(img_tensor)
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        
        # To display
        img_np = np.array(img_resized)                    # HxWx3 uint8 [0-255]
        img_display = img_np / 255.0                       # Normalize to [0,1] for overlay
        pred_rgb = class_to_rgb(pred)                     # Colored mask uint8
        
        # Prepare predicted overlay (semi-transparent, background fully transparent)
        h, w = pred.shape
        pred_rgb_norm = pred_rgb.astype(np.float32) / 255.0
        alpha_pred = np.full((h, w), alpha_value, dtype=np.float32)
        alpha_pred[pred == background_class] = 0.0
        rgba_pred = np.dstack((pred_rgb_norm, alpha_pred))
        
        # Plot columns
        # 0: Input Image
        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f"Input: {fname}")
        axes[i, 0].axis('off')
        
        # 1: Predicted Mask
        axes[i, 1].imshow(pred_rgb)
        axes[i, 1].set_title("Predicted Mask")
        axes[i, 1].axis('off')
        
        # 2: Predicted Overlay
        axes[i, 2].imshow(img_display)
        axes[i, 2].imshow(rgba_pred)
        axes[i, 2].set_title("Predicted Overlay")
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# --- 3. Run it ---
visualize_test_predictions(model, device, n_samples=10)

In [ ]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {num_params:,}")

# Approximate memory size (float32 weights)
model_size_mb = num_params * 4 / (1024 ** 2)
print(f"Approximate model size (float32): {model_size_mb:.2f} MB")

In [ ]:
import torch
import time
import psutil
import os

# ==============================
# 1. Initialize Model
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = YNet(n_channels=3, n_classes=5).to(device)
model.eval()
input_size = (1, 3, 256, 256)
dummy_input = torch.randn(input_size).to(device)

# ==============================
# 2. Parameter Count
# ==============================
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

# ==============================
# 3. Model Size (MB)
# ==============================
param_size  = sum(p.nelement() * p.element_size() for p in model.parameters())
buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
model_size_mb = (param_size + buffer_size) / 1024**2
print(f"Model Size: {model_size_mb:.2f} MB")

# ==============================
# 4. FLOPs Calculation (pure PyTorch hooks)
# ==============================
flop_count = [0]

def conv_flop_hook(module, inp, out):
    # MACs = Cout * Hout * Wout * Cin * Kh * Kw / groups
    # FLOPs = 2 * MACs
    in_tensor  = inp[0]
    batch      = in_tensor.shape[0]
    out_h, out_w = out.shape[2], out.shape[3]
    cin        = module.in_channels
    cout       = module.out_channels
    kh, kw     = module.kernel_size
    groups     = module.groups
    macs = batch * cout * out_h * out_w * (cin // groups) * kh * kw
    flop_count[0] += 2 * macs

def linear_flop_hook(module, inp, out):
    in_tensor = inp[0]
    batch     = in_tensor.shape[0]
    flop_count[0] += 2 * batch * module.in_features * module.out_features

def bn_flop_hook(module, inp, out):
    # 2 ops per element (subtract mean, divide std)
    flop_count[0] += 2 * inp[0].numel()

hooks = []
for m in model.modules():
    if isinstance(m, torch.nn.Conv2d):
        hooks.append(m.register_forward_hook(conv_flop_hook))
    elif isinstance(m, torch.nn.Linear):
        hooks.append(m.register_forward_hook(linear_flop_hook))
    elif isinstance(m, (torch.nn.BatchNorm2d, torch.nn.BatchNorm1d)):
        hooks.append(m.register_forward_hook(bn_flop_hook))

with torch.no_grad():
    model(dummy_input)

for h in hooks:
    h.remove()

print(f"FLOPs: {flop_count[0] / 1e9:.2f} GFLOPs")

# ==============================
# 5. Inference Time & FPS
# ==============================
iterations = 100

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy_input)

if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.time()
for _ in range(iterations):
    with torch.no_grad():
        _ = model(dummy_input)

if device.type == "cuda":
    torch.cuda.synchronize()

end_time  = time.time()
avg_time  = (end_time - start_time) / iterations
fps       = 1.0 / avg_time
print(f"Average Inference Time: {avg_time * 1000:.2f} ms")
print(f"FPS: {fps:.2f}")

# ==============================
# 6. CPU & RAM Usage
# ==============================
process   = psutil.Process(os.getpid())
cpu_usage = psutil.cpu_percent(interval=1)
ram_usage = process.memory_info().rss / 1024**2
print(f"CPU Usage: {cpu_usage:.2f}%")
print(f"RAM Usage: {ram_usage:.2f} MB")

# ==============================
# 7. GPU Memory (if available)
# ==============================
if device.type == "cuda":
    gpu_mem = torch.cuda.max_memory_allocated() / 1024**2
    print(f"GPU Memory Usage: {gpu_mem:.2f} MB")

# ==============================
# 8. Manual Layer-wise Summary
# ==============================
print(f"\n{'='*70}")
print(f"{'Layer':<40} {'Output Shape':<20} {'Params':>10}")
print(f"{'='*70}")

shape_log = {}

def shape_hook(name):
    def hook(module, inp, out):
        if isinstance(out, torch.Tensor):
            shape_log[name] = tuple(out.shape)
    return hook

shape_hooks = []
for name, module in model.named_modules():
    if len(list(module.children())) == 0:   # leaf modules only
        shape_hooks.append(
            module.register_forward_hook(shape_hook(name))
        )

with torch.no_grad():
    model(dummy_input)

for h in shape_hooks:
    h.remove()

total_layer_params = 0
for name, module in model.named_modules():
    if len(list(module.children())) == 0:
        p     = sum(x.numel() for x in module.parameters())
        shape = shape_log.get(name, "N/A")
        print(f"{name:<40} {str(shape):<20} {p:>10,}")
        total_layer_params += p

print(f"{'='*70}")
print(f"{'Total':<40} {'':<20} {total_layer_params:>10,}")

In [ ]:
# Cell 11: Visualize Test Images | Predicted Mask | Predicted Overlay
import os
import random
import matplotlib.pyplot as plt
import torch
import numpy as np
from PIL import Image
from torchvision import transforms

# --- 0. Load the saved model ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Recreate the model architecture
model = YNet(n_channels=3, n_classes=5).to(device)

# Load the saved weights
model.load_state_dict(torch.load('/kaggle/working/best_mars_segmentation_model.pth'))
model.eval()
print("✅ Model loaded successfully from /kaggle/working/best_mars_segmentation_model.pth")

# --- 1. Setup ---
test_image_dir = r"/kaggle/input/datasets/arjunu312003/test-500"
file_names = [f for f in os.listdir(test_image_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"Found {len(file_names)} test images")

# Reuse the inverse color map from before
INV_COLOR_MAP = {v: k for k, v in COLOR_MAP.items()}

def class_to_rgb(mask):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in INV_COLOR_MAP.items():
        rgb[mask == cls] = color
    return rgb

# Image transform (must match training)
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# --- 2. Visualization function ---
def visualize_test_predictions(model, device, n_samples=4):
    model.eval()
    samples = random.sample(file_names, min(n_samples, len(file_names)))
    
    # Changed to 3 columns: Input | Predicted Mask | Predicted Overlay
    fig, axes = plt.subplots(n_samples, 3, figsize=(15, 4 * n_samples))
    
    # Handle single sample case
    if n_samples == 1:
        axes = axes.reshape(1, -1)
    
    # Overlay settings
    alpha_value = 0.6
    background_class = 0
    
    for i, fname in enumerate(samples):
        # Load and preprocess image
        path = os.path.join(test_image_dir, fname)
        img_pil = Image.open(path).convert("RGB")
        img_resized = img_pil.resize((256, 256))
        img_tensor = transform(img_pil).unsqueeze(0).to(device)
        
        # Forward pass
        with torch.no_grad():
            logits = model(img_tensor)
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        
        # To display
        img_np = np.array(img_resized)
        img_display = img_np / 255.0
        pred_rgb = class_to_rgb(pred)
        
        # Prepare predicted overlay (semi-transparent, background fully transparent)
        h, w = pred.shape
        pred_rgb_norm = pred_rgb.astype(np.float32) / 255.0
        alpha_pred = np.full((h, w), alpha_value, dtype=np.float32)
        alpha_pred[pred == background_class] = 0.0
        rgba_pred = np.dstack((pred_rgb_norm, alpha_pred))
        
        # Plot columns
        # 0: Input Image
        axes[i, 0].imshow(img_np)
        axes[i, 0].axis('off')
        
        # 1: Predicted Mask
        axes[i, 1].imshow(pred_rgb)
        axes[i, 1].set_title("Predicted Mask", fontsize=10)
        axes[i, 1].axis('off')
        
        # 2: Predicted Overlay
        axes[i, 2].imshow(img_display)
        axes[i, 2].imshow(rgba_pred)
        axes[i, 2].set_title("Predicted Overlay", fontsize=10)
        axes[i, 2].axis('off')
        
        # Print statistics for each image
        
    
    plt.tight_layout()
    plt.show()

# --- 3. Run it ---
visualize_test_predictions(model, device, n_samples=10)